# Purple Box Factory — Detection, Tracking, and Counting

**Objective:** Train an object detector for healthy and defective boxes, track each physical box across video frames, and generate one ordered status sequence per test video.

The submission protocol allows up to six boxes per video. Status `0` represents a healthy box, status `1` represents a defective box, and `-1` is used for unused positions.

## Pipeline

1. Set up the environment and retrieve the dataset.
2. Inspect and prepare the YOLO-format training data.
3. Fine-tune a pretrained YOLO11 detector.
4. Run object detection on sampled video frames.
5. Associate detections across frames and count each track once when it crosses the exit line.
6. Aggregate class predictions per track, order boxes by crossing time, and export `submission.csv`.

> **Important:** The tracking thresholds and exit-line position are configurable heuristics. They should be tuned using representative videos with known box trajectories. The notebook does not claim a verified competition score.

## 1. Environment and imports

The notebook is designed for Google Colab or a compatible Python environment. A CUDA GPU is recommended for model training and video inference.

In [1]:
# Install only the packages required by this notebook.
%pip install -q ultralytics gdown pyyaml opencv-python-headless tqdm scipy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rbo 0.1.3 requires numpy<2.0,>=1.18, but you have numpy 2.5.5 which is incompatible.
paddlex 3.7.2 requires numpy<2.4,>=1.24, but you have numpy 2.5.5 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from collections import Counter
import os
import random
import shutil
import zipfile
import warnings

import cv2
import gdown
import numpy as np
import pandas as pd
import torch
import yaml
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"Device: {'CUDA' if DEVICE == 0 else 'CPU'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/home/norbert/.venvs/ml/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.7.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Device: CUDA
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


## 2. Download and extract the dataset

The archive is downloaded from the Google Drive file supplied with the problem statement. If the dataset is already available locally, set `DOWNLOAD_DATASET = False` and update `DATASET_DIR`.

The extraction step is idempotent: an existing extracted directory is reused.

In [3]:
DRIVE_FILE_ID = "1yDOCT8WbfaHjXR79wp2yT7-rLNq0cNWD"
DOWNLOAD_DATASET = True

WORK_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
ARCHIVE_PATH = WORK_DIR / "box-factory-dataset.zip"
EXTRACT_DIR = WORK_DIR / "box-factory-dataset"

if DOWNLOAD_DATASET and not EXTRACT_DIR.exists():
    gdown.download(
        f"https://drive.google.com/uc?id={DRIVE_FILE_ID}",
        str(ARCHIVE_PATH),
        quiet=False,
    )
    if not ARCHIVE_PATH.exists() or ARCHIVE_PATH.stat().st_size == 0:
        raise RuntimeError("Dataset download failed. Check the Drive link and access permissions.")

    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE_PATH, "r") as archive:
        archive.extractall(EXTRACT_DIR)

if not EXTRACT_DIR.exists():
    raise FileNotFoundError(
        f"Dataset directory not found: {EXTRACT_DIR}. "
        "Set EXTRACT_DIR or DOWNLOAD_DATASET according to your environment."
    )

print(f"Dataset directory: {EXTRACT_DIR}")

Dataset directory: /home/norbert/Projects/computer_vision_oly/object_tracking/box_factory/box-factory-dataset


## 3. Locate and inspect the training data

The problem description specifies YOLO labels with two classes:

- `0`: `healthy_box`
- `1`: `defective_box`

This section locates the training image and label directories instead of assuming that the archive has one exact nesting layout. It supports either:

- an already split dataset (`images/train`, `images/val`, `labels/train`, `labels/val`), or
- a flat image/label pair (`images`, `labels`), which is split deterministically into training and validation subsets.

The original test videos are expected under `test/videos/raw`.

In [4]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CLASS_NAMES = {0: "healthy_box", 1: "defective_box"}

def find_training_root(search_root: Path) -> Path:
    candidates = [search_root] + [p for p in search_root.rglob("*") if p.is_dir()]
    for candidate in candidates:
        image_dir = candidate / "images"
        label_dir = candidate / "labels"
        if image_dir.is_dir() and label_dir.is_dir():
            has_images = any(p.suffix.lower() in IMAGE_EXTENSIONS for p in image_dir.rglob("*"))
            if has_images:
                return candidate
    raise FileNotFoundError("Could not locate a training directory containing images/ and labels/.")

TRAIN_ROOT = find_training_root(EXTRACT_DIR)
print(f"Training root: {TRAIN_ROOT}")

def count_files(directory: Path, suffixes=None) -> int:
    if not directory.exists():
        return 0
    files = directory.rglob("*") if directory.is_dir() else []
    return sum(1 for p in files if p.is_file() and (suffixes is None or p.suffix.lower() in suffixes))

for split_name in ("train", "val"):
    image_dir = TRAIN_ROOT / "images" / split_name
    label_dir = TRAIN_ROOT / "labels" / split_name
    if image_dir.exists() or label_dir.exists():
        print(
            f"{split_name:>5}: "
            f"{count_files(image_dir, IMAGE_EXTENSIONS)} images, "
            f"{count_files(label_dir, {'.txt'})} label files"
        )

flat_images = TRAIN_ROOT / "images"
flat_labels = TRAIN_ROOT / "labels"
print(
    "Flat layout:",
    count_files(flat_images, IMAGE_EXTENSIONS),
    "images;",
    count_files(flat_labels, {".txt"}),
    "label files",
)

Training root: /home/norbert/Projects/computer_vision_oly/object_tracking/box_factory/box-factory-dataset/train
train: 85 images, 85 label files
  val: 15 images, 15 label files
Flat layout: 100 images; 100 label files


## 4. Prepare a YOLO dataset configuration

If train/validation folders already exist, they are used directly. Otherwise, image files are paired with same-stem YOLO label files and split into an 80/20 train/validation partition.

Missing label files are treated as empty-label images only when no matching file exists; malformed label rows and unsupported class IDs are reported as errors. This makes dataset problems visible before training.

In [5]:
PREPARED_DIR = WORK_DIR / "prepared-box-dataset"
SPLIT_DIRS_EXIST = (
    (TRAIN_ROOT / "images" / "train").is_dir()
    and (TRAIN_ROOT / "images" / "val").is_dir()
    and (TRAIN_ROOT / "labels" / "train").is_dir()
    and (TRAIN_ROOT / "labels" / "val").is_dir()
)

if SPLIT_DIRS_EXIST:
    DATASET_FOR_TRAINING = TRAIN_ROOT
    print("Using the existing train/validation split.")
else:
    image_files = sorted(
        p for p in (TRAIN_ROOT / "images").iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not image_files:
        raise FileNotFoundError(
            "No flat images were found. The archive may use a different layout; inspect TRAIN_ROOT."
        )

    label_source = TRAIN_ROOT / "labels"
    paired_images = []
    for image_path in image_files:
        label_path = label_source / f"{image_path.stem}.txt"
        if not label_path.exists():
            warnings.warn(f"Missing label file for {image_path.name}; an empty label file will be created.")
        paired_images.append((image_path, label_path))

    random.Random(SEED).shuffle(paired_images)
    split_index = max(1, int(len(paired_images) * 0.8))
    if len(paired_images) > 1:
        split_index = min(split_index, len(paired_images) - 1)

    if PREPARED_DIR.exists():
        shutil.rmtree(PREPARED_DIR)

    for split_name, pairs in (
        ("train", paired_images[:split_index]),
        ("val", paired_images[split_index:]),
    ):
        image_out = PREPARED_DIR / "images" / split_name
        label_out = PREPARED_DIR / "labels" / split_name
        image_out.mkdir(parents=True, exist_ok=True)
        label_out.mkdir(parents=True, exist_ok=True)

        for image_path, label_path in pairs:
            shutil.copy2(image_path, image_out / image_path.name)
            destination_label = label_out / f"{image_path.stem}.txt"
            if label_path.exists():
                shutil.copy2(label_path, destination_label)
            else:
                destination_label.write_text("", encoding="utf-8")

    DATASET_FOR_TRAINING = PREPARED_DIR
    print(f"Created deterministic 80/20 split in: {PREPARED_DIR}")

# Validate YOLO label rows.
label_files = list((DATASET_FOR_TRAINING / "labels").rglob("*.txt"))
invalid_rows = []
for label_file in label_files:
    for line_number, line in enumerate(label_file.read_text(encoding="utf-8").splitlines(), start=1):
        fields = line.split()
        if not fields:
            continue
        try:
            values = [float(value) for value in fields]
            class_id = int(values[0])
            valid = (
                len(values) == 5
                and values[0].is_integer()
                and class_id in CLASS_NAMES
                and all(0.0 <= value <= 1.0 for value in values[1:])
                and values[3] > 0
                and values[4] > 0
            )
        except (ValueError, IndexError):
            valid = False
        if not valid:
            invalid_rows.append((str(label_file), line_number, line))

if invalid_rows:
    preview = "\n".join(map(str, invalid_rows[:10]))
    raise ValueError(f"Found {len(invalid_rows)} invalid YOLO label rows. First examples:\n{preview}")

DATA_YAML_PATH = WORK_DIR / "box_factory_data.yaml"
data_config = {
    "path": str(DATASET_FOR_TRAINING),
    "train": "images/train",
    "val": "images/val",
    "names": CLASS_NAMES,
}
with DATA_YAML_PATH.open("w", encoding="utf-8") as file:
    yaml.safe_dump(data_config, file, sort_keys=False, allow_unicode=True)

print(DATA_YAML_PATH.read_text(encoding="utf-8"))
print(f"Training images: {count_files(DATASET_FOR_TRAINING / 'images' / 'train', IMAGE_EXTENSIONS)}")
print(f"Validation images: {count_files(DATASET_FOR_TRAINING / 'images' / 'val', IMAGE_EXTENSIONS)}")

Using the existing train/validation split.
path: /home/norbert/Projects/computer_vision_oly/object_tracking/box_factory/box-factory-dataset/train
train: images/train
val: images/val
names:
  0: healthy_box
  1: defective_box

Training images: 85
Validation images: 15


## 5. Train the object detector

A pretrained YOLO11 small model (`yolo11s.pt`) is fine-tuned on the prepared two-class dataset. The default image size and batch size are intentionally moderate for GPUs with limited VRAM; adjust them only after checking available memory.

The best checkpoint is saved under `runs/detect/box_factory/weights/best.pt`.

In [6]:
TRAIN_MODEL = True

MODEL_NAME = "yolo11s.pt"
EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 4 if torch.cuda.is_available() else 2
PATIENCE = 20

RUNS_DIR = WORK_DIR / "runs" / "detect"
RUN_NAME = "box_factory"
BEST_WEIGHTS = RUNS_DIR / RUN_NAME / "weights" / "best.pt"

if TRAIN_MODEL:
    detector = YOLO(MODEL_NAME)
    detector.train(
        data=str(DATA_YAML_PATH),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        patience=PATIENCE,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        workers=2,
        seed=SEED,
        plots=True,
        verbose=True,
    )
else:
    print("Training skipped. An existing checkpoint will be used.")

if not BEST_WEIGHTS.exists():
    raise FileNotFoundError(
        f"Best model checkpoint not found at {BEST_WEIGHTS}. "
        "Run training or update BEST_WEIGHTS to a valid checkpoint."
    )

print(f"Best checkpoint: {BEST_WEIGHTS}")

New https://pypi.org/project/ultralytics/8.4.215 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.115 🚀 Python-3.12.3 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5771MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/norbert/Projects/computer_vision_oly/object_tracking/box_factory/box_factory_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0

## 6. Validate the trained detector

This validation step reports standard detection metrics on the held-out validation split. These metrics are useful for model selection, but they do not directly measure the competition's video-level exact-match score, which also depends on tracking and counting.

In [7]:
detector = YOLO(str(BEST_WEIGHTS))
validation_metrics = detector.val(
    data=str(DATA_YAML_PATH),
    imgsz=IMAGE_SIZE,
    device=DEVICE,
    verbose=False,
)
print(validation_metrics)

Ultralytics 8.4.115 🚀 Python-3.12.3 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5771MiB)
YOLO11s summary (fused): 101 layers, 9,413,574 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3491.5±381.7 MB/s, size: 145.2 KB)
val: Scanning /home/norbert/Projects/computer_vision_oly/object_tracking/box_factory/box-factory-dataset/train/labels/val.cache... 15 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 15/15 7.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 5.9it/s 0.2s
                   all         15         52      0.998          1      0.995      0.954
Speed: 0.2ms preprocess, 4.3ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /home/norbert/Projects/computer_vision_oly/object_tracking/box_factory/runs/detect/val
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.me

## 7. Configure video inference and tracking

The camera view is treated as a horizontal conveyor: boxes move from left to right, and a box is counted when its tracked center crosses the exit line.

The tracker uses a Hungarian assignment on center-point distances between current detections and active tracks. A track is counted only once. Class predictions are accumulated using detection confidence, which reduces the impact of a single unstable frame.

### Parameters to verify against the actual videos

- `FRAME_STRIDE`: number of frames skipped between detector calls.
- `EXIT_X_RATIO`: horizontal position of the counting line, as a fraction of frame width.
- `MAX_MATCH_DISTANCE`: maximum center displacement allowed between sampled frames.
- `MAX_MISSED_FRAMES`: how many sampled frames a track can remain unmatched.
- `MIN_TRACK_HITS`: minimum matched detections required before a track can be counted.

These are operational parameters, not values proven optimal by the provided notebook.

In [8]:
VIDEOS_DIR = EXTRACT_DIR / "test" / "videos" / "raw"
OUTPUT_CSV = WORK_DIR / "submission.csv"

CONFIDENCE_THRESHOLD = 0.30
NMS_IOU_THRESHOLD = 0.45
FRAME_STRIDE = 2

EXIT_X_RATIO = 0.90
MAX_MATCH_DISTANCE = 90.0
MAX_MISSED_FRAMES = 8
MIN_TRACK_HITS = 3
MAX_BOXES_PER_VIDEO = 6

VIDEO_EXTENSIONS = {".webm", ".mp4", ".avi", ".mov", ".mkv"}

if not VIDEOS_DIR.exists():
    candidates = list(EXTRACT_DIR.rglob("raw"))
    candidates = [p for p in candidates if p.is_dir() and any(
        f.suffix.lower() in VIDEO_EXTENSIONS for f in p.iterdir() if f.is_file()
    )]
    if len(candidates) == 1:
        VIDEOS_DIR = candidates[0]
    else:
        raise FileNotFoundError(f"Could not locate test/videos/raw. Checked: {VIDEOS_DIR}")

video_paths = sorted(
    p for p in VIDEOS_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
)
if not video_paths:
    raise FileNotFoundError(f"No supported videos found in {VIDEOS_DIR}")

print(f"Video directory: {VIDEOS_DIR}")
print(f"Videos found: {len(video_paths)}")

Video directory: /home/norbert/Projects/computer_vision_oly/object_tracking/box_factory/box-factory-dataset/test/videos/raw
Videos found: 30


## 8. Track detections across frames

Each detection contains a bounding box, its center point, predicted class, and confidence. The tracker associates detections to existing tracks using minimum-cost assignment on Euclidean center distance.

A track stores confidence-weighted class scores across its lifetime. Its final class is the class with the larger accumulated score.

In [9]:
class CenterTracker:
    def __init__(self, max_distance=MAX_MATCH_DISTANCE, max_missed=MAX_MISSED_FRAMES):
        self.max_distance = float(max_distance)
        self.max_missed = int(max_missed)
        self.tracks = {}
        self.next_track_id = 0

    @staticmethod
    def _center(bbox):
        x1, y1, x2, y2 = bbox
        return np.array([(x1 + x2) / 2.0, (y1 + y2) / 2.0], dtype=float)

    def update(self, detections):
        # detections: list of dictionaries with bbox, center, class_id, confidence
        track_ids = list(self.tracks.keys())
        matched_detection_indices = set()
        matched_track_ids = set()

        if track_ids and detections:
            cost = np.full((len(track_ids), len(detections)), 1e6, dtype=float)
            for row, track_id in enumerate(track_ids):
                predicted_center = self.tracks[track_id]["center"]
                for col, detection in enumerate(detections):
                    distance = float(np.linalg.norm(predicted_center - detection["center"]))
                    if distance <= self.max_distance:
                        cost[row, col] = distance

            rows, cols = linear_sum_assignment(cost)
            for row, col in zip(rows, cols):
                if cost[row, col] > self.max_distance:
                    continue
                track_id = track_ids[row]
                detection = detections[col]
                track = self.tracks[track_id]

                track["bbox"] = detection["bbox"]
                track["center"] = detection["center"]
                track["hits"] += 1
                track["missed"] = 0
                track["class_scores"][detection["class_id"]] += detection["confidence"]

                matched_track_ids.add(track_id)
                matched_detection_indices.add(col)

        # Create a new track for each unmatched detection.
        for detection_index, detection in enumerate(detections):
            if detection_index in matched_detection_indices:
                continue
            track_id = self.next_track_id
            self.next_track_id += 1
            class_scores = {0: 0.0, 1: 0.0}
            class_scores[detection["class_id"]] += detection["confidence"]
            self.tracks[track_id] = {
                "bbox": detection["bbox"],
                "center": detection["center"],
                "hits": 1,
                "missed": 0,
                "class_scores": class_scores,
                "counted": False,
                "crossing_frame": None,
            }
            matched_track_ids.add(track_id)

        # Age unmatched tracks and remove tracks that have been missing too long.
        expired = []
        for track_id, track in self.tracks.items():
            if track_id not in matched_track_ids:
                track["missed"] += 1
                if track["missed"] > self.max_missed:
                    expired.append(track_id)
        for track_id in expired:
            del self.tracks[track_id]

    @staticmethod
    def final_class(track):
        return max(track["class_scores"], key=track["class_scores"].get)

## 9. Process one video

For each sampled frame, the detector produces box candidates and the tracker updates its active tracks. A track is recorded when its center moves from the left side of the exit line to the right side.

The crossing frame is used to establish the order in which boxes leave the camera view. Each track can contribute at most one result.

In [10]:
def process_video(video_path: Path, model: YOLO):
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f"OpenCV could not open video: {video_path}")

    frame_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    if frame_width <= 0:
        capture.release()
        raise RuntimeError(f"Invalid video dimensions: {video_path}")

    exit_x = frame_width * EXIT_X_RATIO
    tracker = CenterTracker()
    counted_tracks = []
    frame_index = 0

    try:
        while True:
            success, frame = capture.read()
            if not success:
                break

            if frame_index % FRAME_STRIDE != 0:
                frame_index += 1
                continue

            result = model.predict(
                source=frame,
                conf=CONFIDENCE_THRESHOLD,
                iou=NMS_IOU_THRESHOLD,
                device=DEVICE,
                verbose=False,
            )[0]

            detections = []
            if result.boxes is not None:
                for box in result.boxes:
                    x1, y1, x2, y2 = box.xyxy[0].detach().cpu().numpy().tolist()
                    class_id = int(box.cls[0].item())
                    confidence = float(box.conf[0].item())

                    if class_id not in CLASS_NAMES:
                        continue

                    bbox = (float(x1), float(y1), float(x2), float(y2))
                    detections.append({
                        "bbox": bbox,
                        "center": CenterTracker._center(bbox),
                        "class_id": class_id,
                        "confidence": confidence,
                    })

            tracker.update(detections)

            for track_id, track in tracker.tracks.items():
                if track["counted"] or track["hits"] < MIN_TRACK_HITS:
                    continue

                center_x = float(track["center"][0])
                if center_x >= exit_x:
                    track["counted"] = True
                    track["crossing_frame"] = frame_index
                    counted_tracks.append({
                        "frame": frame_index,
                        "track_id": track_id,
                        "class_id": CenterTracker.final_class(track),
                    })

            frame_index += 1
    finally:
        capture.release()

    counted_tracks.sort(key=lambda item: (item["frame"], item["track_id"]))
    return [item["class_id"] for item in counted_tracks], counted_tracks

## 10. Run inference on all test videos

The video list is sorted to keep the output deterministic. Errors are collected and reported instead of silently converting failed videos into predictions of zero boxes.

If more than six tracks are counted, the notebook keeps the first six in crossing order and emits a warning. Such a case should be investigated because the task states that each video contains at most six boxes.

In [11]:
detector = YOLO(str(BEST_WEIGHTS))

submission_rows = []
processing_errors = []
count_distribution = Counter()
detailed_results = {}

for video_path in tqdm(video_paths, desc="Processing test videos"):
    video_id = video_path.stem
    try:
        statuses, track_details = process_video(video_path, detector)
        detailed_results[video_id] = track_details

        if len(statuses) > MAX_BOXES_PER_VIDEO:
            warnings.warn(
                f"{video_id}: {len(statuses)} tracks counted; "
                f"truncating to the first {MAX_BOXES_PER_VIDEO}."
            )
            statuses = statuses[:MAX_BOXES_PER_VIDEO]

        count_distribution[len(statuses)] += 1
        padded_statuses = statuses + [-1] * (MAX_BOXES_PER_VIDEO - len(statuses))
        submission_rows.append([video_id, *padded_statuses])

    except Exception as error:
        processing_errors.append((video_id, repr(error)))
        print(f"Failed to process {video_id}: {error}")

if processing_errors:
    error_summary = "\n".join(f"- {video_id}: {error}" for video_id, error in processing_errors)
    raise RuntimeError(
        "One or more videos could not be processed. Fix the errors before submitting:\n"
        + error_summary
    )

columns = ["video_id"] + [str(index) for index in range(1, MAX_BOXES_PER_VIDEO + 1)]
submission_df = pd.DataFrame(submission_rows, columns=columns)
submission_df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved: {OUTPUT_CSV}")
print(f"Videos processed: {len(submission_df)}")
print("\nBox-count distribution:")
for box_count in sorted(count_distribution):
    print(f"  {box_count} boxes: {count_distribution[box_count]} videos")

display(submission_df)

Processing test videos:   0%|          | 0/30 [00:00<?, ?it/s]

/tmp/ipykernel_778477/1438716512.py:15: UserWarning: factory_test_001_raw: 7 tracks counted; truncating to the first 6.
  warnings.warn(
/tmp/ipykernel_778477/1438716512.py:15: UserWarning: factory_test_002_raw: 7 tracks counted; truncating to the first 6.
  warnings.warn(
/tmp/ipykernel_778477/1438716512.py:15: UserWarning: factory_test_004_raw: 7 tracks counted; truncating to the first 6.
  warnings.warn(
/tmp/ipykernel_778477/1438716512.py:15: UserWarning: factory_test_007_raw: 7 tracks counted; truncating to the first 6.
  warnings.warn(
/tmp/ipykernel_778477/1438716512.py:15: UserWarning: factory_test_010_raw: 7 tracks counted; truncating to the first 6.
  warnings.warn(
/tmp/ipykernel_778477/1438716512.py:15: UserWarning: factory_test_011_raw: 8 tracks counted; truncating to the first 6.
  warnings.warn(
/tmp/ipykernel_778477/1438716512.py:15: UserWarning: factory_test_012_raw: 7 tracks counted; truncating to the first 6.
  warnings.warn(
/tmp/ipykernel_778477/1438716512.py:15: U

Saved: /home/norbert/Projects/computer_vision_oly/object_tracking/box_factory/submission.csv
Videos processed: 30

Box-count distribution:
  6 boxes: 30 videos


,video_id,1,2,3,4,5,6
0,factory_test_001_raw,0,0,1,0,1,0
1,factory_test_002_raw,0,1,1,1,0,0
2,factory_test_003_raw,0,1,0,0,0,1
3,factory_test_004_raw,0,0,1,1,1,1
4,factory_test_005_raw,0,1,1,1,1,0
5,factory_test_006_raw,0,1,0,1,0,0
6,factory_test_007_raw,0,0,1,0,1,1
7,factory_test_008_raw,0,1,0,1,1,0
8,factory_test_009_raw,0,1,0,1,1,1
9,factory_test_010_raw,0,1,1,0,0,0


## 11. Submission checks

The following checks verify the required CSV schema, uniqueness of video IDs, valid class values, and the `-1` padding convention. They cannot verify whether the predictions match the hidden ground truth.

In [12]:
expected_columns = ["video_id", "1", "2", "3", "4", "5", "6"]

assert list(submission_df.columns) == expected_columns
assert submission_df["video_id"].is_unique
assert len(submission_df) == len(video_paths)

status_values = submission_df[expected_columns[1:]].to_numpy()
assert np.isin(status_values, [-1, 0, 1]).all(), "Unexpected status value in submission."

for _, row in submission_df.iterrows():
    values = row[expected_columns[1:]].astype(int).tolist()
    seen_padding = False
    for value in values:
        if value == -1:
            seen_padding = True
        elif seen_padding:
            raise AssertionError(
                f"{row['video_id']}: a box status appears after padding."
            )

print("Submission structure checks passed.")
print(submission_df.head())

Submission structure checks passed.
               video_id  1  2  3  4  5  6
0  factory_test_001_raw  0  0  1  0  1  0
1  factory_test_002_raw  0  1  1  1  0  0
2  factory_test_003_raw  0  1  0  0  0  1
3  factory_test_004_raw  0  0  1  1  1  1
4  factory_test_005_raw  0  1  1  1  1  0


## 12. Package the submission

The competition requires `submission.zip` to contain exactly:

- `notebook.ipynb`
- `submission.csv`

In Colab, save or download this notebook as `notebook.ipynb` before running this cell. If the current notebook is stored under another path, update `NOTEBOOK_PATH` accordingly.

In [13]:
NOTEBOOK_PATH = WORK_DIR / "notebook.ipynb"
SUBMISSION_CSV_PATH = OUTPUT_CSV
ZIP_PATH = WORK_DIR / "submission.zip"

if not NOTEBOOK_PATH.exists():
    print(
        f"Notebook file not found at {NOTEBOOK_PATH}. "
        "Save/download this notebook as notebook.ipynb, then rerun this cell."
    )
elif not SUBMISSION_CSV_PATH.exists():
    raise FileNotFoundError(f"Submission CSV not found: {SUBMISSION_CSV_PATH}")
else:
    with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(NOTEBOOK_PATH, arcname="notebook.ipynb")
        archive.write(SUBMISSION_CSV_PATH, arcname="submission.csv")

    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        print("Archive contents:", archive.namelist())
    print(f"Created: {ZIP_PATH}")

Notebook file not found at /home/norbert/Projects/computer_vision_oly/object_tracking/box_factory/notebook.ipynb. Save/download this notebook as notebook.ipynb, then rerun this cell.


## 13. Notes on interpretation and further tuning

- The detector predicts box locations and per-frame classes; it does not identify physical boxes across time by itself.
- The tracker is a lightweight center-distance association method. It is not a production-grade multi-object tracker and may need stronger motion or appearance cues if boxes overlap or move irregularly.
- The counting line assumes left-to-right motion. If the actual videos move in the opposite direction, the crossing condition and line location must be reversed.
- The class decision uses confidence-weighted votes over the detections assigned to a track. This is generally more stable than trusting only the first or last frame, but it should be checked against labeled examples.
- The final competition metric is exact video-level match. A useful validation protocol is therefore to annotate a small set of representative videos and measure both exact count accuracy and exact ordered status accuracy—not only detector mAP.